Notebook réservé aux regressions (précédemment sur R, adapté sur Python avec statsmodel) 

In [57]:
import pandas as pd
import statsmodels.formula.api as smf
import os
import numpy as np

# chemin relatif qui respecte l'arborescence du Git (pour rendre réplicable)
file_path = os.path.join("..", "data", "merged", "data_regression_std.csv") # prendre les données standardisées!

# Chargement
df = pd.read_csv(file_path)



REGRESSION MCO 

In [58]:

# Régression Linéaire (OLS)

# Adapter la formule pour changer les variables Y ou explicatives X si besoin 

formula = "score_gauche_2020 ~ score_gauche_2014 + log_pop_2020 + log_med_19 + pct_sup_2020"

print(f"Régression sur la formule : {formula}")

# On lance le modèle (missing='drop' ignore juste les lignes avec des NaN résiduels s'il y en a)
model = smf.ols(formula=formula, data=df, missing='drop').fit(cov_type='HC1')
    
# summary du modèle 
print(model.summary())



Régression sur la formule : score_gauche_2020 ~ score_gauche_2014 + log_pop_2020 + log_med_19 + pct_sup_2020
                            OLS Regression Results                            
Dep. Variable:      score_gauche_2020   R-squared:                       0.174
Model:                            OLS   Adj. R-squared:                  0.172
Method:                 Least Squares   F-statistic:                     99.52
Date:                Sat, 20 Dec 2025   Prob (F-statistic):           2.38e-75
Time:                        13:03:20   Log-Likelihood:                -7098.4
No. Observations:                1533   AIC:                         1.421e+04
Df Residuals:                    1528   BIC:                         1.423e+04
Df Model:                           4                                         
Covariance Type:                  HC1                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
---------------

tableau 1 au dessus: piégeux (instabilité de l'offre politique locale, des listes peuvent apparaitre ou disparaitre). Solution: code ci dessous qui ne regarde que les communes avec score gauche >0 en 2014 ET en 2020 (au prix de perdre bcp de lignes de données)

In [59]:

# Chargement des données standardisées

file_path = os.path.join("..", "data", "merged", "data_regression_std.csv")


df = pd.read_csv(file_path)
print(f"Données chargées : {len(df)} communes.")


#  Le test filtrage "Technique vs Politique" 

# On ne garde que les communes où il y a eu une liste de gauche score>0 les deux années.

# Cela élimine le bruit technique des "disparitions" ou "apparitions" de listes.

df_stable = df[(df['score_gauche_2014'] > 0) & (df['score_gauche_2020'] > 0)].copy()

print(f"FILTRAGE:")
print(f"Communes totales : {len(df)}")
print(f"Communes avec structure gauche stable (Test) : {len(df_stable)}")
print(f"Perte de {len(df) - len(df_stable)} communes instables.")

# Régression sur l'offre Stable 
# Formule finale validée (avec les proportions pct_ et les logs)
formula = "score_gauche_2020 ~ score_gauche_2014 + log_pop_2020 + log_med_19 + pct_sup_2020"

print(f"\n RÉSULTATS DU TEST")
print(f"Formule : {formula}")


model_stable = smf.ols(formula=formula, data=df_stable, missing='drop').fit(cov_type='HC1')
print(model_stable.summary())
    
# interprétation automatique
coeff_2014 = model_stable.params['score_gauche_2014']
print("\n VERDICT du test")
if coeff_2014 > 0:
    print(f" Coefficient 2014 positif ({coeff_2014:.3f}) : C'était bien un biais technique!")
    print("Les bastions de 2014 restent des bastions. La corrélation est rétablie.")
else: 
    print(f" Coefficient 2014 toujours négatif ({coeff_2014:.3f}) : C'est un fait POLITIQUE.")
    print("Les bastions de 2014 continuent de s'effondrer structurellement, même à offre constante. Montée du RN?")



Données chargées : 1533 communes.
FILTRAGE:
Communes totales : 1533
Communes avec structure gauche stable (Test) : 371
Perte de 1162 communes instables.

 RÉSULTATS DU TEST
Formule : score_gauche_2020 ~ score_gauche_2014 + log_pop_2020 + log_med_19 + pct_sup_2020
                            OLS Regression Results                            
Dep. Variable:      score_gauche_2020   R-squared:                       0.360
Model:                            OLS   Adj. R-squared:                  0.353
Method:                 Least Squares   F-statistic:                     49.78
Date:                Sat, 20 Dec 2025   Prob (F-statistic):           1.95e-33
Time:                        13:03:20   Log-Likelihood:                -1572.7
No. Observations:                 371   AIC:                             3155.
Df Residuals:                     366   BIC:                             3175.
Df Model:                           4                                         
Covariance Type:         

On remarque aussi que le score 2014 prend tout le poids! 

Méthodoligie et résultats: synthèse de ce notebook et du notebook préparation des données

Analyse de régression visant à expliquer l'évolution du vote de gauche (2014-2020 aux municipales) par des déterminants sociologiques. Voici les points clés de la démarche :


1. Pré-traitement et standardisation

Transformation logarithmique : Les variables de richesse et de population ont été passées au logarithme avant toute standardisation pour éviter la création de valeurs manquantes (NaN) sur des valeurs centrées-réduites négatives

Choix des Variables : Utilisation stricte de taux (%) (Cadres, Diplômés) plutôt que de valeurs absolues pour éviter une multicolinéarité forte avec la variable de population.

Standardisation (Z-Score) : Appliquée uniquement aux variables explicatives (X) pour comparer les coefficients (coeffs beta), mais pas à la variable cible (Y = score gauche année X ou delta score) pour conserver l'interprétation en points de pourcentage de voix.



2. Le "paradoxe" du score 2014 (Biais Technique vs Politique)

Observation initiale : Sur l'échantillon global (~1500 communes), le coefficient du score 2014 était négatif (-0.21). Cela suggérait un effondrement structurel des bastions historiques de gauche.

Hypothèse du Biais : Ce résultat était faussé par l'instabilité de l'offre politique locale (apparition/disparition de listes créant des scores de 0 artificiels).

Correction : En filtrant uniquement sur les communes avec une offre stable (présence de liste de gauche en 2014 ET 2020), le coefficient repasse à +0.67.

Conclusion : L'autocorrélation est rétablie. L'inertie du vote existe toujours ; l'anomalie était TECHNIQUE et non POLITIQUE.



3. Interprétation du Modèle Final (Offre Stable, N=371, R^2=0.36)

Le modèle final est robuste et confirme les dynamiques sociologiques attendues :

Inertie forte, le vote passé reste le premier déterminant du vote futur.

Effet Métropole (log_pop), Plus la commune est peuplée, plus le vote gauche résiste/progresse.

Clivage Culturel (pct_sup), À richesse égale, une forte part de diplômés favorise le vote de gauche (effet "bobo" / vote culturel).

Clivage Économique (log_med) : La richesse reste un frein massif au vote de gauche (coefficient négatif fort).

In [60]:
import pandas as pd
import statsmodels.formula.api as smf
import os
import numpy as np


# regression sur données présidentielles 2017 et 2022. 

# chemin relatif qui respecte l'arborescence du Git (pour rendre réplicable)
file_path_pres = os.path.join("..", "data", "merged", "data_regression_std_presidentielles.csv") # prendre les données standardisées!

# Chargement
df_pres = pd.read_csv(file_path_pres)

In [64]:
# Régression Linéaire (OLS)

# Adapter la formule pour changer les variables Y ou explicatives X si besoin 

formula = "score_gauche_pres_2022 ~ score_gauche_pres_2017 + log_pop_2022 + log_med_19 + pct_sup_2022"

print(f"Régression sur la formule : {formula}")

# On lance le modèle (missing='drop' ignore juste les lignes avec des NaN résiduels s'il y en a)
model = smf.ols(formula=formula, data=df_pres, missing='drop').fit(cov_type='HC1')
    
# summary du modèle 
print(model.summary())

Régression sur la formule : score_gauche_pres_2022 ~ score_gauche_pres_2017 + log_pop_2022 + log_med_19 + pct_sup_2022
                              OLS Regression Results                              
Dep. Variable:     score_gauche_pres_2022   R-squared:                       0.737
Model:                                OLS   Adj. R-squared:                  0.737
Method:                     Least Squares   F-statistic:                 1.325e+04
Date:                    Tue, 23 Dec 2025   Prob (F-statistic):               0.00
Time:                            14:01:35   Log-Likelihood:                -86444.
No. Observations:                   30717   AIC:                         1.729e+05
Df Residuals:                       30712   BIC:                         1.729e+05
Df Model:                               4                                         
Covariance Type:                      HC1                                         
                             coef    std err       

Pistes analyse économétrique:

regression niveau (score_gauche_pres_2022 ~ score_gauche_pres_2017 + log_pop_2022 + log_med_19 + pct_sup_2022) : l'inertie reste le premier facteur, le clivage culturel domine, et la richesse reste un frein. 

regression dynamique deltas: voir suite


In [66]:
# Régression Linéaire (OLS)

# Adapter la formule pour changer les variables Y ou explicatives X si besoin 

formula = "delta_score_gauche_pres ~ + croissance_pop_17_22 + log_med_19 + delta_pct_diplome_17_22"

print(f"Régression sur la formule : {formula}")

# On lance le modèle (missing='drop' ignore juste les lignes avec des NaN résiduels s'il y en a)
model = smf.ols(formula=formula, data=df_pres, missing='drop').fit(cov_type='HC1')
    
# summary du modèle 
print(model.summary())

Régression sur la formule : delta_score_gauche_pres ~ + croissance_pop_17_22 + log_med_19 + delta_pct_diplome_17_22
                               OLS Regression Results                              
Dep. Variable:     delta_score_gauche_pres   R-squared:                       0.060
Model:                                 OLS   Adj. R-squared:                  0.060
Method:                      Least Squares   F-statistic:                     602.3
Date:                     Tue, 23 Dec 2025   Prob (F-statistic):               0.00
Time:                             15:04:30   Log-Likelihood:                -89227.
No. Observations:                    30717   AIC:                         1.785e+05
Df Residuals:                        30713   BIC:                         1.785e+05
Df Model:                                3                                         
Covariance Type:                       HC1                                         
                              coef    std er

1. Changement de signe pour le revenu (log_med_19) : Contrairement au modèle de niveau, le coefficient devient positif (+0,94). La gauche progresse désormais plus vite dans les communes les plus aisées (embourgeoisement du vote). c'est exactement ce qu'on cherchait à montrer avec la problématique de base

2. Moteur éducatif (delta_pct_diplome) : L'augmentation locale du niveau de diplôme est un levier direct de croissance pour la gauche (+0,34). 

3. Dynamique démographique (croissance_pop) : Les communes en forte croissance favorisent le bloc de gauche (+0,29), suggérant une "importation" de nouveaux électeurs de gauche. (les nouveaux arrivants sont souvent des jeunes actifs ou diplomés, qui importent un vote de gauche)

4. Poussée politique (Intercept) :  à sociologie constante, la gauche progresse de +1,17 point, reflétant une dynamique politique nationale favorable entre 2017 et 2022.

Colinéarité : Le Cond. No. est excellent (1,21). L'absence de multicolinéarité garantit que chaque variable apporte une information propre et que les coefficients sont mathématiquement stables. 

a faire: 
1) comparer significativité local vs national
2) isoler score Jadot et regarder les différences coeffs (notamment signe coeff revenu median)

3) entrainer un RandomForest pour explorer l'importance des variables
4) cartographie des résidus pour voir où le modèle a échoué pour la corrélation spatiale
5) réfléchir à des variables omises historiques et culturelles que l'Insee ne capte pas avec la socio-démo.